# HIL-SERL in simulation (Colab)
Trains HIL-SERL (RLPD with demos and human corrections) on the simulated Franka cube-lift task. A scripted expert stands in for the human: it records the demos and takes over in some training episodes.

**Runtime:** *Runtime → Change runtime type*, then *Runtime → Run all*.
- **L4 GPU** (Colab Pro): best value. **A100 GPU**: fastest, uses more compute units. **T4 GPU**: works on the free tier, but the learner is about as fast as a laptop RTX 3060.
- Do not pick G4 (Blackwell): this setup uses jax 0.4.20 with CUDA 11 / cuDNN 8, which predates Blackwell.
- Turn on **High RAM**: the replay buffer is then raised from 20k to 50k transitions.

**If the session times out or disconnects:** open the notebook again and *Run all*. Everything that matters lives in Google Drive (`MyDrive/hilserl-runs/<RUN_NAME>`): checkpoints, the actor's replay-buffer dumps, the demos and the episode log. Training continues from the latest checkpoint, so at most about 1,000 steps are lost. Keep `RUN_NAME` the same to resume; change it to start a new run.

Drive space: about 2.5 GB per run (last 2 checkpoints of ~0.3 GB, last 10 buffer dumps of ~0.19 GB, demos 0.24 GB). Deleted old checkpoints go to the Drive trash, so empty it now and then.

In [ ]:
import subprocess
name, mem_mib, cc = [x.strip() for x in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"], text=True
).splitlines()[0].split(",")]
GPU_MEM_GB = int(mem_mib) / 1024
ram_gb = int(open("/proc/meminfo").read().split()[1]) / 1024**2
print(f"GPU: {name}, {GPU_MEM_GB:.0f} GB, compute capability {cc}; RAM: {ram_gb:.0f} GB")
if float(cc) >= 10:
    raise SystemExit(f"{name} is a Blackwell GPU (sm_{cc.replace('.', '')}). This setup pins jax 0.4.20 with CUDA 11 / cuDNN 8, "
                     "which cannot run on it. Switch the runtime to L4 or A100 (or T4).")

## 1. Settings and Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

RUN_NAME = "run1"            # keep the same name to resume after a disconnect
TARGET_ENV_STEPS = 30000     # stop when the actor has taken this many env steps
INTERVENE_PROB = 0.2         # fraction of episodes where the scripted expert takes over

DRIVE_ROOT = "/content/drive/MyDrive/hilserl-runs"
WORKDIR = "/content/hilserl-work"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ.update(
    WORKDIR=WORKDIR,
    RUN_DIR=f"/content/runs/{RUN_NAME}",          # fast local disk; copied to Drive by run_training.sh
    SYNC_DIR=f"{DRIVE_ROOT}/{RUN_NAME}",
    TARGET_ENV_STEPS=str(TARGET_ENV_STEPS),
    HILSERL_SIM_INTERVENE_PROB=str(INTERVENE_PROB),
    # GPU memory caps (fractions of the card): more headroom on small cards
    LEARNER_FRAC=".5" if GPU_MEM_GB >= 23 else ".6",
    ACTOR_FRAC=".2" if GPU_MEM_GB >= 23 else ".25",
    # replay buffer capacity: ~0.1 MB per transition, so 50k needs ~5 GB of RAM (High RAM runtimes)
    HILSERL_SIM_BUFFER="50000" if ram_gb >= 40 else "20000",
    MUJOCO_GL="egl",
)
print("learner/actor GPU fractions:", os.environ["LEARNER_FRAC"], os.environ["ACTOR_FRAC"], "| replay buffer:", os.environ["HILSERL_SIM_BUFFER"])
print("run folder on Drive:", os.environ["SYNC_DIR"], "(exists)" if os.path.isdir(os.environ["SYNC_DIR"]) else "(new)")

## 2. Install (about 5 minutes)
Clones this repository, then `hilserl/setup.sh` clones `rail-berkeley/hil-serl` and `rail-berkeley/serl` at pinned commits, adds the `sim_pick_cube` task and installs a Python 3.10 environment (jax 0.4.20 with CUDA 11, cuDNN 8.9).

In [ ]:
%%bash
set -e
apt-get -qq install -y libegl1 libgl1 rsync > /dev/null
[ -d /content/seminar ] || git clone -q https://github.com/yigitbatki/seminar.git /content/seminar
git -C /content/seminar pull -q   # pick up fixes when the cell is re-run
bash /content/seminar/hilserl/setup.sh "$WORKDIR" 2>&1 | tail -n 5

## 3. Check the GPU and the simulator

In [ ]:
%%bash
source "$WORKDIR/.venv/bin/activate"
cd "$WORKDIR/hil-serl/examples"
python - <<'PY' 2>&1 | grep -vE "Warning|warn|Gym has been|gymnasium|migration"
import os, time, numpy as np, jax
print("jax devices:", jax.devices())
os.environ["HILSERL_SIM_MODE"] = "demo"
from experiments.mappings import CONFIG_MAPPING
env = CONFIG_MAPPING["sim_pick_cube"]().get_environment(fake_env=False)
obs, _ = env.reset()
print({k: v.shape for k, v in obs.items()})
t0, n, wins = time.time(), 0, 0
for _ in range(3):
    obs, _ = env.reset(); done = False
    while not done:
        obs, r, done, _, info = env.step(np.zeros(7)); n += 1
    wins += info["succeed"]
print(f"expert {wins}/3, {n / (time.time() - t0):.1f} env steps/s")
PY

## 4. Demos (recorded once, kept in Drive)

In [ ]:
import glob, shutil, subprocess
demo_drive = f"{DRIVE_ROOT}/demos"
os.makedirs(demo_drive, exist_ok=True)
found = sorted(glob.glob(f"{demo_drive}/sim_pick_cube_20_demos_*.pkl"))
if not found:
    subprocess.run(
        f'source {WORKDIR}/.venv/bin/activate && cd {WORKDIR}/hil-serl/examples && '
        'N_DEMOS=20 bash experiments/sim_pick_cube/run_record_demos.sh 2>&1 | tail -n 2',
        shell=True, executable="/bin/bash", check=True)
    new = sorted(glob.glob(f"{WORKDIR}/hil-serl/examples/demo_data/sim_pick_cube_20_demos_*.pkl"))[-1]
    shutil.copy(new, demo_drive)
    found = sorted(glob.glob(f"{demo_drive}/sim_pick_cube_20_demos_*.pkl"))
os.environ["DEMO_PATH"] = found[-1]
print("demos:", found[-1])

## 5. Train (resumes automatically)
Runs the learner and the actor in the background and prints progress every 30 s. The run is copied to Drive after every checkpoint and at least every 10 minutes. The cell ends when `TARGET_ENV_STEPS` is reached. To stop early, interrupt the cell (■); the latest state is saved to Drive first.

"policy successes (no takeover)" counts only episodes where the expert did not step in.

In [ ]:
import subprocess, sys
p = subprocess.Popen(["bash", "/content/seminar/hilserl/run_training.sh"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in p.stdout:
        print(line, end="")
except KeyboardInterrupt:
    p.send_signal(2); p.wait()  # run_training.sh stops both processes and syncs to Drive

## 6. Evaluate the latest checkpoint
20 episodes with the policy alone (no expert takeovers).

In [ ]:
%%bash
source "$WORKDIR/.venv/bin/activate"
cd "$WORKDIR/hil-serl/examples"
STEP=$(ls -d "$SYNC_DIR"/checkpoint_* | grep -oE '[0-9]+$' | sort -n | tail -1)
echo "checkpoint $STEP"
yes "" | HILSERL_SIM_INTERVENE_PROB=0 HILSERL_SIM_EPISODE_LOG= CKPT="$SYNC_DIR" XLA_PYTHON_CLIENT_MEM_FRACTION=.3 \
  bash experiments/sim_pick_cube/run_actor.sh --eval_checkpoint_step=$STEP --eval_n_trajs=20 2>&1 \
  | grep -E "success rate|average time" | tail -n 2

## 7. Learning curve (re-run any time, also while training runs in another tab)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f"{os.environ['SYNC_DIR']}/episodes.csv")
df["env_steps"] = df["length"].cumsum()
own = df[df["intervened_steps"] == 0]
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax.plot(own["env_steps"], own["succeed"].rolling(20, min_periods=1).mean(), label="policy-alone success (rolling 20)")
ax.plot(df["env_steps"], (df["intervened_steps"] > 0).rolling(20, min_periods=1).mean(), alpha=.5, label="intervention rate (rolling 20)")
ax.set_xlabel("env steps"); ax.set_ylabel("rate"); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=.3)
ax2.plot(own["env_steps"], own["length"].rolling(20, min_periods=1).mean())
ax2.set_xlabel("env steps"); ax2.set_ylabel("episode length, policy alone (rolling 20)"); ax2.grid(alpha=.3)
plt.show()
print(f"{len(df)} episodes, {df['length'].sum()} env steps, policy-alone success in the last 20: {own['succeed'].tail(20).mean():.2f}")